# FinGPT CoT + PoT：v3 Mixed Program SFT 实验流程

本 notebook 实现新的 **CoT + PoT** 训练路线。
 - CoT SFT 学习 `Reasoning + Answer`
 - Program SFT 学习 `Evidance + Program`

最终数值答案不再由模型直接生成，而是由 executor 执行 `Program` 得到。这比 v2 的 `Evidence + Program + Answer + Normalized Answer` 更符合当前主任务：**FinQA/ConvFinQA 的可验证金融数值推理**。

---

当前已有 v3 SFT2 quick benchmark 信号：

```text
v3 sft2 executed_answer_accuracy = 0.875
pass@1 = 0.875
pass@8 = 0.9375
gap = 0.0625
```

这说明继续堆两阶段 SFT 的边际收益可能有限，甚至可能让模型过度贴合 gold Program 字符串，压窄采样空间。因此本 notebook 默认采用更克制的路线：

```text
Base
-> Fino1 / FinCoT CoT cold-start
-> FinQA + ConvFinQA turn-level mixed v3 Program SFT
-> benchmark against v2 sft2_dual_merged and existing v3 sft2_program_merged
-> 若 pass@k 仍有空间，再把 checkpoint 交给 GRPO
```

核心原则：

- **v3 是主线**：最终指标看 `execute(Program) == answer_norm`。
- **v2 是 baseline**：保留 `sft2_dual_merged` 作为对照，不再作为新路线 schema。
- **CoT 不污染 Program**：Fino1/FinCoT 只训练 `Reasoning + Answer`，不输出 `Program: N/A`。
- **mixed SFT 是默认**：FinQA 和 ConvFinQA 直接混合，避免继续拉长 SFT 链条。
- **two-stage 只是 ablation**：需要复现既有 v3 思路时再打开。


## 环境准备

这一节只检查当前 Python / CUDA 环境，并提供可选依赖安装开关。

默认所有训练和 benchmark 都不会自动运行。Notebook 中的训练、merge、评估都由 `RUN_*` 开关控制：

```python
RUN_COT_SFT = False
RUN_MERGE_COT = False
RUN_PROGRAM_SFT = False
RUN_MERGE_PROGRAM = False
RUN_BENCHMARK = False
```

推荐运行顺序：

1. 先运行环境与配置 cell。
2. 生成并审计 CoT cold-start 数据。
3. 生成并审计 v3 Program mixed 数据。
4. 确认 target schema 正确后，再打开训练开关。
5. 训练完成并 merge 后，再打开 benchmark。


In [ ]:


try:
    import torch
    print('torch =', torch.__version__)
    print('cuda =', torch.version.cuda)
    print('cuda available =', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu =', torch.cuda.get_device_name(0))
except Exception as exc:
    print('torch check skipped:', repr(exc))


cwd = /root/FinQA
torch = 2.8.0+cu128
cuda = 12.8
cuda available = True
gpu = NVIDIA GeForce RTX 4090


In [ ]:
# Optional installs. Keep disabled unless the environment is missing dependencies.
RUN_INSTALL_DEPS = False
RUN_INSTALL_MODELSCOPE = False

if RUN_INSTALL_DEPS:
    subprocess.run(['python', '-m', 'pip', 'install', '-r', 'requirements.txt', '--upgrade'], check=True)
if RUN_INSTALL_MODELSCOPE:
    subprocess.run(['python', '-m', 'pip', 'install', 'modelscope'], check=True)


In [6]:
import os
HF_ENDPOINT = "https://hf-mirror.com"
os.environ["HF_ENDPOINT"] = HF_ENDPOINT
print("HF_ENDPOINT:", os.getenv("HF_ENDPOINT"))

HF_ENDPOINT: https://hf-mirror.com


## 实验配置

这里定义本 notebook 的关键实验选择。

默认配置：

```python
PROGRAM_SFT_MODE = "mixed"
USE_COT_COLD_START = True
USE_FINO1 = True
USE_FINCOT = True
FINO1_RATIO = 0.8
FINCOT_RATIO = 0.2
PROGRAM_CONVFINQA_TO_FINQA_RATIO = 2.0
PROGRAM_SFT_EPOCHS = 1
PROGRAM_SFT_LR = "5e-6"
```

含义：

- `PROGRAM_SFT_MODE="mixed"`：默认把 FinQA replay 和 ConvFinQA turn-level 直接混合训练。
- `PROGRAM_SFT_MODE="two_stage"`：只作为对照实验，走 `FinQA SFT1 -> ConvFinQA+FinQA replay SFT2`。
- `FINO1_RATIO=0.8`：CoT cold-start 以 Fino1 Reasoning Path FinQA 为主，因为它更贴近 FinQA Program 主线。
- `FINCOT_RATIO=0.2`：FinCoT 只做低比例泛金融 reasoning 补充。
- `PROGRAM_CONVFINQA_TO_FINQA_RATIO=2.0`：mixed Program SFT 中 ConvFinQA 是主体，FinQA replay 用来防止单轮表文能力遗忘。

输出目录独立于 v2/v3：

```text
/root/autodl-tmp/data/financial_reasoning_cot_pot
/root/autodl-tmp/outputs/financial_reasoning_cot_pot
```

这样不会覆盖现有 v2/v3 实验产物。


In [ ]:
from pathlib import Path
import json
import os
import random
import shutil
import subprocess
from typing import Any, Dict, Iterable, List, Optional, Tuple

PROJECT_ROOT = Path('/root/FinQA')
os.chdir(PROJECT_ROOT)
print('cwd =', Path.cwd())

BASE_MODEL = Path('/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct')
TEMPLATE_NAME = 'qwen'
SEED = 42
random.seed(SEED)

PROGRAM_SFT_MODE = 'mixed'  # default: 'mixed'; optional: 'two_stage'
SFT_VARIANT = 'program_executor_sft'
STRICT_TIERS = 'A'
CONVFINQA_MODE = 'turn_level'
FILTER_CONFLICTING_PROMPTS = True

USE_COT_COLD_START = True
USE_FINO1 = True
USE_FINCOT = True
FINO1_RATIO = 0.8
FINCOT_RATIO = 0.2
MAX_COT_ROWS = 6000
MAX_FINO1_ROWS = None
MAX_FINCOT_ROWS = None

FINO1_DATASET = 'TheFinAI/Fino1_Reasoning_Path_FinQA'
FINO1_SPLIT = 'train'
FINCOT_DATASET = 'TheFinAI/FinCoT'
FINCOT_SPLIT = 'SFT'

# Prefer local files prepared ahead of time. Each path may be a single file or a directory.
# Supported local formats: .jsonl, .json, .csv, .parquet, .arrow.
FINO1_LOCAL_PATH = Path('/root/autodl-tmp/data/thefinai/Fino1_Reasoning_Path_FinQA')
FINCOT_LOCAL_PATH = Path('/root/autodl-tmp/data/thefinai/FinCoT')
ALLOW_HF_FALLBACK = False

PROGRAM_CONVFINQA_TO_FINQA_RATIO = 2.0
VALIDATION_CONVFINQA_ROWS = 307
VALIDATION_FINQA_ROWS = 153
PROGRAM_SFT_EPOCHS = 1
PROGRAM_SFT_LR = '5e-6'
PROGRAM_MODEL_MAX_LENGTH = '1024'  # use 1536 if memory allows

COT_SFT_EPOCHS = 1
COT_SFT_LR = '1e-5'
COT_MODEL_MAX_LENGTH = '1024'

RUN_DOWNLOAD_COT = False  # default: load pre-downloaded local TheFinAI files
RUN_BUILD_PROGRAM_DATA = True
RUN_COT_SFT = False
RUN_MERGE_COT = False
RUN_PROGRAM_SFT = False
RUN_MERGE_PROGRAM = False
RUN_BENCHMARK = False

DISK_ROOT = Path('/root/autodl-tmp')
DATA_DIR = DISK_ROOT / 'data' / 'financial_reasoning_cot_pot'
OUTPUT_ROOT = DISK_ROOT / 'outputs' / 'financial_reasoning_cot_pot'
RAW_CACHE_DIR = DISK_ROOT / 'data' / 'financial_reasoning' / 'raw'

COT_DIR = DATA_DIR / 'cot_sft'
PROGRAM_DIR = DATA_DIR / 'program_sft'
NORMALIZED_DIR = DATA_DIR / 'normalized'
AUDIT_DIR = DATA_DIR / 'audit'
VALIDATION_DIR = DATA_DIR / 'validation'
REPORT_DIR = DATA_DIR / 'reports'

COT_TRAIN_FILE = COT_DIR / 'train_cot_sft.jsonl'
COT_TRAIN_DIR = COT_DIR / 'train_dir'

# program数据构建
FINQA_PROGRAM_FILE = PROGRAM_DIR / 'train_finqa_program_strict.jsonl'
CONVFINQA_PROGRAM_FILE = PROGRAM_DIR / 'train_convfinqa_turn_program_strict.jsonl'
PROGRAM_MIXED_FILE = PROGRAM_DIR / 'train_program_mixed.jsonl'
PROGRAM_VALID_FILE = VALIDATION_DIR / 'valid_program_mixed.jsonl'
PROGRAM_MIXED_DIR = PROGRAM_DIR / 'train_mixed_dir'
PROGRAM_VALID_DIR = VALIDATION_DIR / 'valid_dir'
SFT1_PROGRAM_DIR = PROGRAM_DIR / 'sft1_dir_program'
SFT2_PROGRAM_DIR = PROGRAM_DIR / 'sft2_dir_program'

# 训练模型输出
COT_OUT = OUTPUT_ROOT / 'cot_sft'
COT_MERGED_OUT = OUTPUT_ROOT / 'cot_sft_merged'
PROGRAM_MIXED_OUT = OUTPUT_ROOT / 'program_mixed'
PROGRAM_MIXED_MERGED_OUT = OUTPUT_ROOT / 'program_mixed_merged'

# 如果分阶段program sft
SFT1_PROGRAM_OUT = OUTPUT_ROOT / 'sft1_program'
SFT1_PROGRAM_MERGED_OUT = OUTPUT_ROOT / 'sft1_program_merged'
SFT2_PROGRAM_OUT = OUTPUT_ROOT / 'sft2_program'
SFT2_PROGRAM_MERGED_OUT = OUTPUT_ROOT / 'sft2_program_merged'

BENCHMARK_DIR = OUTPUT_ROOT / 'benchmarks' / 'cot_pot_program_passk'

# tensorboard --logdir
TB_LOG_DIR = OUTPUT_ROOT / 'tensorboard'

# 已有的baseline
V2_BASELINE = Path('/root/autodl-tmp/outputs/financial_reasoning_v2/sft2_dual_merged')
V3_EXISTING_BASELINE = Path('/root/autodl-tmp/outputs/financial_reasoning_v3/sft2_program_merged')

for d in [
    DATA_DIR, OUTPUT_ROOT, COT_DIR, COT_TRAIN_DIR, PROGRAM_DIR, NORMALIZED_DIR, AUDIT_DIR,
    VALIDATION_DIR, REPORT_DIR, PROGRAM_MIXED_DIR, PROGRAM_VALID_DIR, SFT1_PROGRAM_DIR,
    SFT2_PROGRAM_DIR, COT_OUT, COT_MERGED_OUT, PROGRAM_MIXED_OUT, PROGRAM_MIXED_MERGED_OUT,
    SFT1_PROGRAM_OUT, SFT1_PROGRAM_MERGED_OUT, SFT2_PROGRAM_OUT, SFT2_PROGRAM_MERGED_OUT,
    BENCHMARK_DIR, TB_LOG_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    'program_sft_mode': PROGRAM_SFT_MODE,
    'data_dir': str(DATA_DIR),
    'output_root': str(OUTPUT_ROOT),
    'cot_train_file': str(COT_TRAIN_FILE),
    'program_mixed_file': str(PROGRAM_MIXED_FILE),
    'program_policy_for_rl': str(PROGRAM_MIXED_MERGED_OUT if PROGRAM_SFT_MODE == 'mixed' else SFT2_PROGRAM_MERGED_OUT),
}, ensure_ascii=False, indent=2))


cwd = /root/FinQA
{
  "program_sft_mode": "mixed",
  "data_dir": "/root/autodl-tmp/data/financial_reasoning_cot_pot",
  "output_root": "/root/autodl-tmp/outputs/financial_reasoning_cot_pot",
  "cot_train_file": "/root/autodl-tmp/data/financial_reasoning_cot_pot/cot_sft/train_cot_sft.jsonl",
  "program_mixed_file": "/root/autodl-tmp/data/financial_reasoning_cot_pot/program_sft/train_program_mixed.jsonl",
  "program_policy_for_rl": "/root/autodl-tmp/outputs/financial_reasoning_cot_pot/program_mixed_merged"
}


## 公共工具函数

这一节提供 notebook 内部复用的小工具：

- `read_jsonl` / `write_jsonl`：读写训练数据。
- `sample_records` / `oversample_records`：采样和 replay。
- `assistant_target` / `user_prompt`：从 ShareGPT 格式中读取 prompt 和 target。
- `ensure_single_file_dir`：把 jsonl 放入 `train_file_dir` 风格目录，供 SFT trainer 使用。
- `run_cmd`：打印命令；只有对应 `RUN_*` 为 `True` 时才真正执行。

这些函数不定义训练策略，只服务于后续数据构建、审计、训练命令拼装。


In [6]:
def write_jsonl(path: Path, rows: Iterable[Dict[str, Any]]) -> int:
    path.parent.mkdir(parents=True, exist_ok=True)
    count = 0
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
            count += 1
    return count


def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        return []
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception as exc:
                raise ValueError(f'Invalid JSONL at {path}:{i}: {exc}') from exc
    return rows


def sample_records(rows: List[Dict[str, Any]], target_rows: Optional[int], seed: int) -> List[Dict[str, Any]]:
    if target_rows is None or target_rows >= len(rows):
        out = list(rows)
        random.Random(seed).shuffle(out)
        return out
    return random.Random(seed).sample(rows, target_rows)


def oversample_records(rows: List[Dict[str, Any]], target_rows: int, seed: int) -> List[Dict[str, Any]]:
    if not rows or target_rows <= 0:
        return []
    rng = random.Random(seed)
    out = []
    while len(out) < target_rows:
        chunk = list(rows)
        rng.shuffle(chunk)
        out.extend(chunk)
    return out[:target_rows]


def first_text(value: Any) -> str:
    if value is None:
        return ''
    if isinstance(value, list):
        return '\n'.join(first_text(x) for x in value if first_text(x)).strip()
    return str(value).strip()


def assistant_target(row: Dict[str, Any]) -> str:
    conv = row.get('conversations') or []
    if len(conv) < 2 or not isinstance(conv[1], dict):
        return row.get('response') or row.get('completion') or ''
    return conv[1].get('value') or conv[1].get('content') or ''


def user_prompt(row: Dict[str, Any]) -> str:
    conv = row.get('conversations') or []
    if conv and isinstance(conv[0], dict):
        return conv[0].get('value') or conv[0].get('content') or ''
    return row.get('prompt') or ''


def ensure_single_file_dir(directory: Path, jsonl_file: Path):
    directory.mkdir(parents=True, exist_ok=True)
    target = directory / jsonl_file.name
    shutil.copyfile(jsonl_file, target)
    return target


def run_cmd(cmd: List[str], enabled: bool, cwd: Path = PROJECT_ROOT):
    print(' '.join(str(x) for x in cmd))
    if enabled:
        subprocess.run([str(x) for x in cmd], cwd=str(cwd), check=True)
    else:
        print('[dry-run] set the corresponding RUN_* flag to True to execute')


## CoT Cold-Start

`/root/autodl-tmp/data/financial_reasoning_cot_pot/cot_sft/train_dir` 

**构造来源**

在 [run_fingpt_cot_pot.ipynb](/root/FinQA/run_fingpt_cot_pot.ipynb) 里，`cot_sft/train_dir` 来自 CoT cold-start 数据构建，配置是：

```python
USE_COT_COLD_START = True
USE_FINO1 = True
USE_FINCOT = True
FINO1_RATIO = 0.8
FINCOT_RATIO = 0.2
MAX_COT_ROWS = 6000
SEED = 42
```

实际组成是：

| source_dataset | 行数 | 来源 |
|---|---:|---|
| `fino1_finqa_path` | 4800 | `/root/autodl-tmp/data/thefinai/Fino1_Reasoning_Path_FinQA/train.parquet` |
| `fincot_sft` | 1200 | `/root/autodl-tmp/data/thefinai/FinCoT/SFT-00000-of-00001.parquet` |
| total | 6000 | 80% Fino1 + 20% FinCoT |

- `FINCOT_SPLIT = 'SFT'`，所以构造 `cot_sft` 时用的是 `SFT-00000-of-00001.parquet`。

**样本格式**

每条被转成 ShareGPT 格式：

```json
{
  "conversations": [
    {
      "from": "human",
      "value": "You are a financial reasoning assistant...\n\nQuestion:\n...\n\nOutput format:\nReasoning: ...\n\nAnswer: ..."
    },
    {
      "from": "gpt",
      "value": "Reasoning: ...\n\nAnswer: ..."
    }
  ],
  "metadata": {
    "source_dataset": "...",
    "program_available": false,
    "raw_keys": [...]
  }
}
```

这一阶段是纯 CoT SFT：

```text
Question/context -> Reasoning + Answer
```


**字段抽取逻辑**

notebook 的转换函数会从原始记录里抽这些字段：

- `question`：优先从 `Open-ended Verifiable Question`、`Question`、`prompt`、`query`、`instruction` 等字段取；如果是 chat 格式，则取 user/human 消息。
- `reasoning`：优先从 `Complex_CoT`、`Reasoning_process`、`reasoning`、`rationale`、`cot` 等字段取。
- `answer`：优先从 `Ground-True Answer`、`Answer`、`Final_response`、`final_answer`、`label`、`output` 等字段取。
- `response`：作为 fallback，用于补 reasoning 或 answer。

然后拼成：

```python
prompt = (
    "You are a financial reasoning assistant. Solve the question with concise reasoning.\n\n"
    f"Question:\n{question}\n\n"
    "Output format:\n"
    "Reasoning: ...\n\n"
    "Answer: ..."
)

target = f"Reasoning: {reasoning}\n\nAnswer: {answer}"
```

如果没有 `question` 或没有 `answer`，该条会被跳过。`reasoning` 如果缺失，会用 response fallback；再不行就填一句通用 reasoning 占位。

**采样和混合**

1. 从本地加载 Fino1 和 FinCoT。
2. 分别转换成 ShareGPT CoT 格式。
3. 按 `MAX_COT_ROWS=6000` 和比例采样：
   - Fino1 target = `round(6000 * 0.8) = 4800`
   - FinCoT target = `6000 - 4800 = 1200`
4. 用固定 seed 分别采样：
   - Fino1: `SEED + 13`
   - FinCoT: `SEED + 14`
5. 合并后再用 `SEED + 15` shuffle。
6. 写入 `cot_sft/train_cot_sft.jsonl`。
7. 用 `shutil.copyfile` 复制到 `cot_sft/train_dir/train_cot_sft.jsonl`。

###  数据构建

Fino1/FinCoT provide natural-language reasoning supervision only. The target contains `Reasoning:` and `Answer:`, with no `Program:` field.

本 notebook 默认改为 **本地文件加载**，避免训练环境依赖 Hugging Face 网络。

默认路径：

```text
/root/autodl-tmp/data/thefinai/Fino1_Reasoning_Path_FinQA
/root/autodl-tmp/data/thefinai/FinCoT
```

你可以把变量改成具体文件或目录：

```python
FINO1_LOCAL_PATH = Path('/path/to/fino1_train.jsonl')
FINCOT_LOCAL_PATH = Path('/path/to/fincot_sft.parquet')
```

支持格式：

```text
.jsonl / .json / .csv / .parquet / .arrow
```

如果传入目录，loader 会递归读取该目录下支持的文件，并优先按文件名中的 split 过滤，例如 `train` 或 `SFT`。默认 `ALLOW_HF_FALLBACK=False`，本地文件不存在时只给 warning，不会自动联网下载。确实需要临时走 Hugging Face 时，再手动打开：

```python
ALLOW_HF_FALLBACK = True
RUN_DOWNLOAD_COT = True
```

数据角色：

- Fino1：FinQA 风格 CoT 主数据，用来补公式意图和证据选择解释。
- FinCoT：低比例泛金融 reasoning 增广，用来补多样性。
- 真正的 Program supervision 只来自后续 FinQA/ConvFinQA gold Program。


In [7]:
def make_sharegpt(prompt: str, target: str, metadata: Dict[str, Any]) -> Dict[str, Any]:
    return {
        'conversations': [
            {'from': 'human', 'value': prompt.strip()},
            {'from': 'gpt', 'value': target.strip()},
        ],
        'metadata': metadata,
    }


def build_cot_prompt(question: str) -> str:
    return (
        'You are a financial reasoning assistant. Solve the question with concise reasoning.\n\n'
        f'Question:\n{question.strip()}\n\n'
        'Output format:\n'
        'Reasoning: ...\n\n'
        'Answer: ...'
    )


def build_cot_target(reasoning: str, answer: str, fallback_response: str = '') -> str:
    reasoning = reasoning.strip() or fallback_response.strip() or 'Use the financial evidence and arithmetic implied by the question.'
    answer = answer.strip() or fallback_response.strip()
    return f'Reasoning: {reasoning}\n\nAnswer: {answer}'


SUPPORTED_LOCAL_DATA_EXTS = {'.jsonl', '.json', '.csv', '.parquet', '.arrow'}


def discover_local_data_files(path: Path, split: Optional[str] = None) -> List[Path]:
    path = Path(path)
    if path.is_file():
        return [path] if path.suffix.lower() in SUPPORTED_LOCAL_DATA_EXTS else []
    if not path.exists():
        return []
    files = [p for p in path.rglob('*') if p.is_file() and p.suffix.lower() in SUPPORTED_LOCAL_DATA_EXTS]
    if split:
        split_lower = split.lower()
        split_matches = [p for p in files if split_lower in p.name.lower() or split_lower in str(p.parent).lower()]
        if split_matches:
            files = split_matches
    return sorted(files)


def load_local_rows_from_file(path: Path) -> List[Dict[str, Any]]:
    suffix = path.suffix.lower()
    if suffix == '.jsonl':
        rows = []
        with path.open('r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        return rows
    if suffix == '.json':
        obj = json.loads(path.read_text(encoding='utf-8'))
        if isinstance(obj, list):
            return [dict(x) for x in obj]
        if isinstance(obj, dict):
            for key in ['data', 'train', 'SFT', 'RL', 'rows', 'examples']:
                value = obj.get(key)
                if isinstance(value, list):
                    return [dict(x) for x in value]
            return [obj]
        raise ValueError(f'Unsupported JSON root in {path}: {type(obj)}')
    if suffix in {'.csv', '.parquet', '.arrow'}:
        import pandas as pd
        if suffix == '.csv':
            df = pd.read_csv(path)
        elif suffix == '.parquet':
            df = pd.read_parquet(path)
        else:
            try:
                import pyarrow.ipc as ipc
                with path.open('rb') as f:
                    reader = ipc.RecordBatchFileReader(f)
                    df = reader.read_all().to_pandas()
            except Exception:
                df = pd.read_feather(path)
        return df.to_dict(orient='records')
    raise ValueError(f'Unsupported local data file: {path}')


def load_local_dataset_rows(path: Path, split: Optional[str] = None, dataset_label: str = '') -> List[Dict[str, Any]]:
    files = discover_local_data_files(Path(path), split=split)
    if not files:
        print(f'[warn] no local {dataset_label or "dataset"} files found at {path}')
        return []
    rows = []
    for file in files:
        file_rows = load_local_rows_from_file(file)
        print(f'[local] {dataset_label or "dataset"}: {file} -> {len(file_rows)} rows')
        rows.extend(file_rows)
    return rows


def load_hf_dataset_rows(dataset_name: str, split: str, cache_dir: Path) -> List[Dict[str, Any]]:
    from datasets import load_dataset
    ds = load_dataset(dataset_name, split=split, cache_dir=str(cache_dir))
    return [dict(row) for row in ds]


def load_cot_source_rows(local_path: Path, dataset_name: str, split: str, cache_dir: Path, dataset_label: str) -> List[Dict[str, Any]]:
    rows = load_local_dataset_rows(local_path, split=split, dataset_label=dataset_label)
    if rows:
        return rows
    if ALLOW_HF_FALLBACK and RUN_DOWNLOAD_COT:
        print(f'[hf fallback] {dataset_label}: {dataset_name} split={split}')
        return load_hf_dataset_rows(dataset_name, split, cache_dir)
    print(f'[warn] {dataset_label} skipped. Set local path correctly, or enable ALLOW_HF_FALLBACK=True and RUN_DOWNLOAD_COT=True.')
    return []



def _norm_key(key: Any) -> str:
    return ''.join(ch for ch in str(key).lower() if ch.isalnum())


def get_first_field(rec: Dict[str, Any], candidates: List[str]) -> str:
    if not isinstance(rec, dict):
        return ''
    norm_to_key = {_norm_key(k): k for k in rec.keys()}
    for name in candidates:
        key = norm_to_key.get(_norm_key(name))
        if key is not None:
            value = first_text(rec.get(key))
            if value:
                return value
    return ''


def extract_chat_fields(rec: Dict[str, Any]) -> Tuple[str, str]:
    messages = rec.get('messages') or rec.get('conversations') or rec.get('chat')
    if not isinstance(messages, list):
        return '', ''
    user_parts, assistant_parts = [], []
    for msg in messages:
        if not isinstance(msg, dict):
            continue
        role = str(msg.get('role') or msg.get('from') or '').lower()
        content = first_text(msg.get('content') or msg.get('value') or msg.get('text'))
        if not content:
            continue
        if role in {'user', 'human'}:
            user_parts.append(content)
        elif role in {'assistant', 'gpt'}:
            assistant_parts.append(content)
    return '\n'.join(user_parts).strip(), '\n'.join(assistant_parts).strip()


def split_reasoning_answer(text: str) -> Tuple[str, str]:
    text = first_text(text)
    if not text:
        return '', ''
    markers = ['Final Answer:', 'Final answer:', 'Answer:', 'The answer is']
    for marker in markers:
        if marker in text:
            before, after = text.rsplit(marker, 1)
            answer = (marker + after).strip() if marker.lower().startswith('the answer') else after.strip()
            return before.strip(), answer.strip()
    return text.strip(), text.strip()


def extract_cot_components(rec: Dict[str, Any]) -> Tuple[str, str, str, str]:
    chat_question, chat_response = extract_chat_fields(rec)
    question = get_first_field(rec, [
        'Open-ended Verifiable Question', 'Question', 'question', 'prompt', 'query', 'instruction',
        'problem', 'input', 'user', 'task', 'source',
    ]) or chat_question
    answer = get_first_field(rec, [
        'Ground-True Answer', 'Ground-Truth Answer', 'Ground Truth Answer', 'Answer', 'answer',
        'Final_response', 'Final Response', 'final_response', 'final_answer', 'label', 'output',
    ])
    reasoning = get_first_field(rec, [
        'Complex_CoT', 'Complex CoT', 'Reasoning_process', 'Reasoning Process', 'reasoning_process',
        'reasoning', 'rationale', 'analysis', 'cot', 'chain_of_thought',
    ])
    response = get_first_field(rec, [
        'Response', 'response', 'Final_response', 'Final Response', 'completion', 'chosen', 'assistant',
        'output', 'target',
    ]) or chat_response
    if not reasoning and response:
        parsed_reasoning, parsed_answer = split_reasoning_answer(response)
        reasoning = parsed_reasoning
        if not answer:
            answer = parsed_answer
    if not answer and response:
        _, answer = split_reasoning_answer(response)
    return question, reasoning, answer, response


def convert_generic_cot_rows(rows: List[Dict[str, Any]], source_dataset: str, max_rows: Optional[int], seed: int) -> List[Dict[str, Any]]:
    converted = []
    missing_question = 0
    missing_answer = 0
    for rec in rows:
        question, reasoning, answer, response = extract_cot_components(rec)
        if not question:
            missing_question += 1
            continue
        if not answer:
            missing_answer += 1
            continue
        converted.append(make_sharegpt(
            build_cot_prompt(question),
            build_cot_target(reasoning, answer, response),
            {
                'source_dataset': source_dataset,
                'program_available': False,
                'raw_keys': sorted(str(k) for k in rec.keys())[:40] if isinstance(rec, dict) else [],
            },
        ))
    print(f'[convert] {source_dataset}: raw={len(rows)} converted={len(converted)} missing_question={missing_question} missing_answer={missing_answer}')
    if rows:
        print(f'[convert] {source_dataset}: sample_keys={sorted(str(k) for k in rows[0].keys())[:60]}')
    return sample_records(converted, max_rows, seed)


def convert_fino1_rows(rows: List[Dict[str, Any]], max_rows: Optional[int] = None) -> List[Dict[str, Any]]:
    return convert_generic_cot_rows(rows, 'fino1_finqa_path', max_rows, SEED + 11)


def convert_fincot_rows(rows: List[Dict[str, Any]], max_rows: Optional[int] = None) -> List[Dict[str, Any]]:
    return convert_generic_cot_rows(rows, 'fincot_sft', max_rows, SEED + 12)


def build_cot_dataset():
    fino1_rows, fincot_rows = [], []
    if USE_FINO1:
        fino1_raw = load_cot_source_rows(
            FINO1_LOCAL_PATH,
            FINO1_DATASET,
            FINO1_SPLIT,
            DATA_DIR / 'hf_cache' / 'fino1',
            'Fino1 Reasoning Path FinQA',
        )
        fino1_rows = convert_fino1_rows(fino1_raw, MAX_FINO1_ROWS)
    if USE_FINCOT:
        fincot_raw = load_cot_source_rows(
            FINCOT_LOCAL_PATH,
            FINCOT_DATASET,
            FINCOT_SPLIT,
            DATA_DIR / 'hf_cache' / 'fincot',
            'FinCoT',
        )
        fincot_rows = convert_fincot_rows(fincot_raw, MAX_FINCOT_ROWS)

    if MAX_COT_ROWS is not None and (fino1_rows or fincot_rows):
        fino1_target = int(round(MAX_COT_ROWS * FINO1_RATIO)) if fino1_rows else 0
        fincot_target = MAX_COT_ROWS - fino1_target if fincot_rows else 0
        fino1_rows = sample_records(fino1_rows, min(fino1_target, len(fino1_rows)), SEED + 13)
        fincot_rows = sample_records(fincot_rows, min(fincot_target, len(fincot_rows)), SEED + 14)

    mixed = fino1_rows + fincot_rows
    random.Random(SEED + 15).shuffle(mixed)
    write_jsonl(COT_TRAIN_FILE, mixed)
    ensure_single_file_dir(COT_TRAIN_DIR, COT_TRAIN_FILE)
    return {
        'fino1_local_path': str(FINO1_LOCAL_PATH),
        'fincot_local_path': str(FINCOT_LOCAL_PATH),
        'allow_hf_fallback': ALLOW_HF_FALLBACK,
        'fino1_rows': len(fino1_rows),
        'fincot_rows': len(fincot_rows),
        'total_rows': len(mixed),
        'cot_train_file': str(COT_TRAIN_FILE),
    }

cot_report = build_cot_dataset() if USE_COT_COLD_START else {'skipped': True}
print(json.dumps(cot_report, ensure_ascii=False, indent=2))


[local] Fino1 Reasoning Path FinQA: /root/autodl-tmp/data/thefinai/Fino1_Reasoning_Path_FinQA/train.parquet -> 5499 rows
[convert] fino1_finqa_path: raw=5499 converted=5499 missing_question=0 missing_answer=0
[convert] fino1_finqa_path: sample_keys=['Complex_CoT', 'Ground-True Answer', 'Open-ended Verifiable Question', 'Response']
[local] FinCoT: /root/autodl-tmp/data/thefinai/FinCoT/SFT-00000-of-00001.parquet -> 7686 rows
[convert] fincot_sft: raw=7686 converted=7686 missing_question=0 missing_answer=0
[convert] fincot_sft: sample_keys=['Final_response', 'Negative_reasoning_process', 'Negative_response', 'Question', 'Reasoning_process']
{
  "fino1_local_path": "/root/autodl-tmp/data/thefinai/Fino1_Reasoning_Path_FinQA",
  "fincot_local_path": "/root/autodl-tmp/data/thefinai/FinCoT",
  "allow_hf_fallback": false,
  "fino1_rows": 4800,
  "fincot_rows": 1200,
  "total_rows": 6000,
  "cot_train_file": "/root/autodl-tmp/data/financial_reasoning_cot_pot/cot_sft/train_cot_sft.jsonl"
}


In [8]:
def audit_cot_schema(path: Path) -> Dict[str, Any]:
    rows = read_jsonl(path)
    bad_program = []
    missing_reasoning = []
    missing_answer = []
    for i, row in enumerate(rows[:2000]):
        target = assistant_target(row)
        if 'Program:' in target:
            bad_program.append(i)
        if 'Reasoning:' not in target:
            missing_reasoning.append(i)
        if 'Answer:' not in target:
            missing_answer.append(i)
    return {
        'rows': len(rows),
        'checked_rows': min(len(rows), 2000),
        'program_token_in_target_count': len(bad_program),
        'missing_reasoning_count': len(missing_reasoning),
        'missing_answer_count': len(missing_answer),
        'sample_target': assistant_target(rows[0])[:800] if rows else '',
    }

cot_audit = audit_cot_schema(COT_TRAIN_FILE)
print(json.dumps(cot_audit, ensure_ascii=False, indent=2))
if cot_audit.get('program_token_in_target_count'):
    raise ValueError('CoT cold-start target must not contain Program:')


{
  "rows": 6000,
  "checked_rows": 2000,
  "program_token_in_target_count": 0,
  "missing_reasoning_count": 0,
  "missing_answer_count": 0,
  "sample_target": "Reasoning: So, we have some numbers here about tax rates for 2018. The statutory tax rate is supposed to be 19.0%. That's like the standard baseline rate they're supposed to use. But the effective tax rate, which is what they actually end up paying, is 11.7%. There's definitely a difference there, and it seems like something interesting is happening with the international operations.\n\nLet's break this down. We have something called ‘taxes on international operations’ listed as -7.3. That negative sign probably means it's bringing the overall tax rate down, maybe due to lower tax rates in certain countries or strategic financial setups across the globe. It seems this is a kind of tax break that impacts the effective tax rate significantly.\n\nIf we just look at this from a basic standpoint, w"
}


### 训练

这一阶段是可选的轻量 SFT。

训练目标：

```text
Question -> Reasoning + Answer
```

默认训练参数较克制：

```text
epochs = 1
learning_rate = 1e-5
LoRA rank = 8
model_max_length = 1024
```

默认 `RUN_COT_SFT=False`，因此 cell 只打印命令，不会占用 GPU。若不想做 CoT cold-start，可以跳过这一阶段；后续 Program SFT 会自动回退到 base model。

产物：

```text
cot_sft
cot_sft_merged
```

后续 mixed Program SFT 的 base 优先使用 `cot_sft_merged`；若不存在，则使用 `BASE_MODEL`。


In [ ]:
!python -m training.supervised_finetuning \
--model_name_or_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
--tokenizer_name_or_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
--train_file_dir /root/autodl-tmp/data/financial_reasoning_cot_pot/cot_sft/train_dir \
--validation_split_percentage 1 --do_eval --eval_steps 100 --eval_strategy steps \
--per_device_eval_batch_size 1 --prediction_loss_only True --eval_accumulation_steps 1 --max_eval_samples 32 \
--do_train --use_peft --num_train_epochs 1 --per_device_train_batch_size 1 \
--max_grad_norm 0.5 --gradient_accumulation_steps 16 --gradient_checkpointing True \
--learning_rate 1e-5 --warmup_steps 50 --weight_decay 0.05 \
--logging_steps 10 --save_steps 200 --save_total_limit 2 --logging_first_step True \
--report_to tensorboard --logging_dir /root/autodl-tmp/outputs/financial_reasoning_cot_pot/tensorboard/cot_sft \
--model_max_length 1024 --target_modules q_proj,k_proj,v_proj,o_proj \
--lora_rank 8 --lora_alpha 16 --lora_dropout 0.05 \
--torch_dtype bfloat16 --bf16 --device_map auto \
--output_dir /root/autodl-tmp/outputs/financial_reasoning_cot_pot/cot_sft \
--preprocessing_num_workers 16 --template_name qwen

In [18]:
!python -m tooling.merge_peft_adapter \
--base_model /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
--tokenizer_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
--lora_model /root/autodl-tmp/outputs/financial_reasoning_cot_pot/cot_sft \
--output_dir /root/autodl-tmp/outputs/financial_reasoning_cot_pot/cot_sft_merged

Namespace(base_model='/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct', tokenizer_path='/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct', lora_model='/root/autodl-tmp/outputs/financial_reasoning_cot_pot/cot_sft', resize_emb=False, output_dir='/root/autodl-tmp/outputs/financial_reasoning_cot_pot/cot_sft_merged', hf_hub_model_id='', hf_hub_token=None)
Base model: /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct
LoRA model: /root/autodl-tmp/outputs/financial_reasoning_cot_pot/cot_sft
Loading LoRA for causal language model
Loading weights: 100%|███████████████████████| 339/339 [00:03<00:00, 102.41it/s]
Merging with merge_and_unload...
Saving to Hugging Face format...
Writing model shards: 100%|███████████████████████| 2/2 [00:30<00:00, 15.50s/it]
Done! model saved to /root/autodl-tmp/outputs/financial_reasoning_cot_pot/cot_sft_merged


### quickeval

In [ ]:
# quick eval without CoT answer scoring

! python -m evaluation.evaluate_financial_benchmarks \
  --tokenizer_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
  --model_entry cot_sft=/root/autodl-tmp/outputs/financial_reasoning_cot_pot/cot_sft_merged \
  --finqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/finqa/test.json \
  --convfinqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/convfinqa_turn/dev_turn.json \
  --finqa_max_samples 16 \
  --convfinqa_max_samples 16 \
  --max_new_tokens 1024 \
  --pass_k 1,4,8 \
  --num_samples_per_example 8 \
  --processor_sft_variant program_executor_sft \
  --load_in_4bit \
  --output_dir /root/autodl-tmp/outputs/financial_reasoning_cot_pot/benchmarks/cot_sft

[prep] convfinqa_turn: prepared 421 conversations with 1767 full-history turns
[prep] convfinqa_turn: skipped 627 rows during benchmark example build
[prep] finqa: skipped 464 rows during benchmark example build
[skip] Fineval disabled (use --run_fineval or set --fineval_local_file to enable).
[eval] model=cot_sft path=/root/autodl-tmp/outputs/financial_reasoning_cot_pot/cot_sft_merged adapter=none
Evaluating cot_sft: 100%|███████████████████████| 32/32 [15:40<00:00, 29.38s/it]
[done] Saved benchmark outputs to: /root/autodl-tmp/outputs/financial_reasoning_cot_pot/benchmarks/cot_sft


In [17]:
# quick eval with CoT answer scoring 
! python -m evaluation.evaluate_financial_benchmarks \
  --tokenizer_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
  --model_entry base=/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
  --finqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/finqa/test.json \
  --convfinqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/convfinqa_turn/dev_turn.json \
  --finqa_max_samples 8 \
  --convfinqa_max_samples 8 \
  --max_new_tokens 1024 \
  --pass_k 1,4,8 \
  --num_samples_per_example 8 \
  --processor_sft_variant program_executor_sft \
  --numeric_output_format cot_program \
  --load_in_4bit \
  --output_dir /root/autodl-tmp/outputs/financial_reasoning_cot_pot/benchmarks/base_reasoning_passk

[prep] convfinqa_turn: prepared 421 conversations with 1767 full-history turns
[prep] convfinqa_turn: skipped 627 rows during benchmark example build
[prep] finqa: skipped 464 rows during benchmark example build
[skip] Fineval disabled (use --run_fineval or set --fineval_local_file to enable).
[eval] model=base path=/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct adapter=none
Evaluating base: 100%|██████████████████████████| 16/16 [09:44<00:00, 36.56s/it]
[done] Saved benchmark outputs to: /root/autodl-tmp/outputs/financial_reasoning_cot_pot/benchmarks/base_reasoning_passk


In [ ]:
# quick eval with CoT answer scoring 
! python -m evaluation.evaluate_financial_benchmarks \
  --tokenizer_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
  --model_entry cot=/root/autodl-tmp/outputs/financial_reasoning_cot_pot/cot_sft_merged \
  --finqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/finqa/test.json \
  --convfinqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/convfinqa_turn/dev_turn.json \
  --finqa_max_samples 8 \
  --convfinqa_max_samples 8 \
  --max_new_tokens 1024 \
  --pass_k 1,4,8 \
  --num_samples_per_example 8 \
  --processor_sft_variant program_executor_sft \
  --numeric_output_format cot_program \
  --load_in_4bit \
  --output_dir /root/autodl-tmp/outputs/financial_reasoning_cot_pot/benchmarks/cot_reasoning_passk

## Mixed Program SFT


直接复用 run_fingpt_v3.ipynb 生成的 sft2 数据 `data/financial_reasoning_v3/clean/sft2_dir_program`

下略直接进行训练

### 数据构建
---

这一阶段回到本项目主任务：可验证 Program reasoning。

继续复用现有数据处理入口：

```bash
python -m financial_data_processors   --task sft   --sft_variant program_executor_sft   --strict_tiers A   --filter_conflicting_prompts true
```

生成两个 strict-A Program SFT 文件：

```text
train_finqa_program_strict.jsonl
train_convfinqa_turn_program_strict.jsonl
```

Program target 必须是 v3 schema：

```text
Evidence:
- ...

Program: ...
```

不能包含：

```text
Reasoning:
Answer:
Normalized Answer:
```

答案由 executor 执行 `Program` 后得到，因此训练阶段不让模型继续“心算”最终数值。


In [ ]:

# Optional: rebuild cot_pot-local Program SFT source files.
# The main Program SFT cell below reuses /root/autodl-tmp/data/financial_reasoning_v3/clean/sft2_dir_program.

! python -m financial_data_processors \
  --task sft \
  --dataset_family finqa \
  --source_file /root/autodl-tmp/data/financial_reasoning/raw/finqa/train.json \
  --output_file /root/autodl-tmp/data/financial_reasoning_cot_pot/program_sft/train_finqa_program_strict.jsonl \
  --normalized_output_file /root/autodl-tmp/data/financial_reasoning_cot_pot/normalized/finqa_train_program.jsonl \
  --audit_output_file /root/autodl-tmp/data/financial_reasoning_cot_pot/audit/finqa_train_program_audit.jsonl \
  --sft_variant program_executor_sft \
  --strict_tiers A \
  --filter_conflicting_prompts true

! python -m financial_data_processors \
  --task sft \
  --dataset_family convfinqa_turn \
  --source_file /root/autodl-tmp/data/financial_reasoning/raw/convfinqa_turn/train_turn.json \
  --output_file /root/autodl-tmp/data/financial_reasoning_cot_pot/program_sft/train_convfinqa_turn_program_strict.jsonl \
  --normalized_output_file /root/autodl-tmp/data/financial_reasoning_cot_pot/normalized/convfinqa_train_turn_program.jsonl \
  --audit_output_file /root/autodl-tmp/data/financial_reasoning_cot_pot/audit/convfinqa_train_turn_program_audit.jsonl \
  --sft_variant program_executor_sft \
  --strict_tiers A \
  --filter_conflicting_prompts true \
  --convfinqa_mode turn_level


### FinQA ConvFinQA混合

这一阶段把 FinQA 和 ConvFinQA 合并为默认 Program SFT 训练集。

混合方式：

```text
train_program_mixed.jsonl =
  ConvFinQA turn-level strict-A
  + FinQA replay sampled by ratio
```

默认比例：

```text
ConvFinQA : FinQA replay = 2 : 1
```

为什么合并两阶段 SFT：

- ConvFinQA 是更难的多轮 follow-up 任务，应该从一开始进入主训练分布。
- FinQA replay 负责保留单轮表文能力，不需要单独先训一个完整 SFT1。
- 当前 v3 SFT2 的 `pass@1` 已接近 `pass@8`，继续拉长 SFT 链条可能过拟合 gold Program 字符串。
- 一次 mixed SFT 更适合作为后续 RL 起点，因为输出分布可能没那么窄。

这一节还会做 schema audit：

- CoT 数据中不能出现 `Program:`。
- Program 数据中必须出现 `Evidence:` 和 `Program:`。
- Program 数据中不能出现 `Reasoning:`、`Answer:`、`Normalized Answer:`。


In [ ]:

# Reuse the v3 mixed Program SFT data for cot_pot Program SFT.
# This cell only inspects the existing data summary and one training file sample.

from pathlib import Path
import json

summary_path = Path('/root/autodl-tmp/data/financial_reasoning_v3/clean/train_sft2_program_balanced_summary.json')
train_dir = Path('/root/autodl-tmp/data/financial_reasoning_v3/clean/sft2_dir_program')
train_file = train_dir / 'train_sft2_program_balanced.jsonl'
valid_file = Path('/root/autodl-tmp/data/financial_reasoning_v3/validation/valid_program_balanced.jsonl')

print('Program SFT train dir:', train_dir)
print('Program SFT train file exists:', train_file.exists(), train_file)
print('Program SFT validation file exists:', valid_file.exists(), valid_file)

if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(json.dumps({
        'sft_variant': summary.get('sft_variant'),
        'convfinqa_mode': summary.get('convfinqa_mode'),
        'sft2_train_rows': summary.get('sft2_train_rows'),
        'sft2_train_convfinqa_rows': summary.get('sft2_train_convfinqa_rows'),
        'sft2_train_finqa_replay_rows': summary.get('sft2_train_finqa_replay_rows'),
        'validation_rows': summary.get('validation_rows'),
        'convfinqa_to_finqa_ratio': summary.get('convfinqa_to_finqa_ratio'),
        'train_audit': summary.get('train_audit'),
    }, ensure_ascii=False, indent=2))
else:
    print('Missing summary:', summary_path)

with train_file.open('r', encoding='utf-8') as f:
    sample = json.loads(next(f))
assistant = sample['conversations'][1]['value']
print('sample source_dataset:', sample.get('source_dataset'))
print('sample target preview:')
print(assistant[:800])
assert 'Evidence:' in assistant and 'Program:' in assistant
assert 'Normalized Answer:' not in assistant and 'Answer:' not in assistant and 'Reasoning:' not in assistant


### 训练

默认路线：

```text
program_base = cot_sft_merged if exists else BASE_MODEL
program_base -> mixed Program SFT -> program_mixed -> program_mixed_merged
```

默认训练参数：

```text
epochs = 1
learning_rate = 5e-6
gradient_accumulation_steps = 16
max_grad_norm = 0.5
LoRA rank = 8
target_modules = q_proj,k_proj,v_proj,o_proj
model_max_length = 1024
```

如果显存允许，可以把 `PROGRAM_MODEL_MAX_LENGTH` 改成 `1536`。如果 OOM，保持 `1024`，和现有 v3 notebook 更接近。

如果两阶段训练program sft，这一节只训练 FinQA SFT1；下一节继续训练 SFT2。


In [ ]:
!python -m training.supervised_finetuning \
    --model_name_or_path /root/autodl-tmp/outputs/financial_reasoning_cot_pot/cot_sft_merged \
    --tokenizer_name_or_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
    --train_file_dir /root/autodl-tmp/data/financial_reasoning_v3/clean/sft2_dir_program \
    --validation_split_percentage 1 --do_eval --eval_steps 100 --eval_strategy steps \
    --per_device_eval_batch_size 1 --prediction_loss_only True --eval_accumulation_steps 1 --max_eval_samples 32 \
    --do_train --use_peft \
    --num_train_epochs 1 --per_device_train_batch_size 1 \
    --max_grad_norm 0.5 --gradient_accumulation_steps 16 --gradient_checkpointing True \
    --learning_rate 5e-6 --warmup_steps 50 --weight_decay 0.05 \
    --logging_steps 10 --save_steps 200 --save_total_limit 2 \
    --logging_first_step True --report_to tensorboard \
    --logging_dir /root/autodl-tmp/outputs/financial_reasoning_cot_pot/tensorboard/program_mixed \
    --model_max_length 1536 --target_modules q_proj,k_proj,v_proj,o_proj \
    --lora_rank 8 --lora_alpha 16 --lora_dropout 0.05 \
    --torch_dtype bfloat16 --bf16 \
    --device_map auto \
    --output_dir /root/autodl-tmp/outputs/financial_reasoning_cot_pot/program_mixed \
    --preprocessing_num_workers 16 \
    --template_name qwen



### Merge Program SFT LoRA

Program SFT 训练完成后，把 LoRA 合并到 `cot_sft_merged`，产出最终 cot_pot policy：

```text
/root/autodl-tmp/outputs/financial_reasoning_cot_pot/program_mixed_merged
```


In [19]:

! python -m tooling.merge_peft_adapter \
  --base_model /root/autodl-tmp/outputs/financial_reasoning_cot_pot/cot_sft_merged \
  --tokenizer_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
  --lora_model /root/autodl-tmp/outputs/financial_reasoning_cot_pot/program_mixed \
  --output_dir /root/autodl-tmp/outputs/financial_reasoning_cot_pot/program_mixed_merged


Namespace(base_model='/root/autodl-tmp/outputs/financial_reasoning_cot_pot/cot_sft_merged', tokenizer_path='/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct', lora_model='/root/autodl-tmp/outputs/financial_reasoning_cot_pot/program_mixed', resize_emb=False, output_dir='/root/autodl-tmp/outputs/financial_reasoning_cot_pot/program_mixed_merged', hf_hub_model_id='', hf_hub_token=None)
Base model: /root/autodl-tmp/outputs/financial_reasoning_cot_pot/cot_sft_merged
LoRA model: /root/autodl-tmp/outputs/financial_reasoning_cot_pot/program_mixed
Loading LoRA for causal language model
Loading weights: 100%|███████████████████████| 339/339 [00:03<00:00, 110.93it/s]
Merging with merge_and_unload...
Saving to Hugging Face format...
Writing model shards: 100%|███████████████████████| 2/2 [00:28<00:00, 14.43s/it]
Done! model saved to /root/autodl-tmp/outputs/financial_reasoning_cot_pot/program_mixed_merged


In [20]:
! python -m evaluation.evaluate_financial_benchmarks \
  --tokenizer_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
  --model_entry cot=/root/autodl-tmp/outputs/financial_reasoning_cot_pot/program_mixed_merged \
  --finqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/finqa/test.json \
  --convfinqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/convfinqa_turn/dev_turn.json \
  --finqa_max_samples 8 \
  --convfinqa_max_samples 8 \
  --max_new_tokens 1024 \
  --pass_k 1,4,8 \
  --num_samples_per_example 8 \
  --processor_sft_variant program_executor_sft \
  --numeric_output_format cot_program \
  --load_in_4bit \
  --output_dir /root/autodl-tmp/outputs/financial_reasoning_cot_pot/benchmarks/program_reasoning_passk

[prep] convfinqa_turn: prepared 421 conversations with 1767 full-history turns
[prep] convfinqa_turn: skipped 627 rows during benchmark example build
[prep] finqa: skipped 464 rows during benchmark example build
[skip] Fineval disabled (use --run_fineval or set --fineval_local_file to enable).
[eval] model=cot path=/root/autodl-tmp/outputs/financial_reasoning_cot_pot/program_mixed_merged adapter=none
Evaluating cot: 100%|███████████████████████████| 16/16 [09:36<00:00, 36.02s/it]
[done] Saved benchmark outputs to: /root/autodl-tmp/outputs/financial_reasoning_cot_pot/benchmarks/program_reasoning_passk


## Benchmark：和 v2/v3 baseline 对比

评估目标不是只看 loss，而是看生成式 Program 是否真的可执行、执行后答案是否正确。

默认对比：

```text
base
v2_sft2 = /root/autodl-tmp/outputs/financial_reasoning_v2/sft2_dual_merged
v3_sft2 = /root/autodl-tmp/outputs/financial_reasoning_v3/sft2_program_merged
cot_pot = /root/autodl-tmp/outputs/financial_reasoning_cot_pot/program_mixed_merged
```

核心指标：

```text
executed_answer_accuracy
program_execution_rate
program_string_accuracy
pass@1_greedy
pass@4
pass@8
avg_prediction_chars
```

判断规则：

- 如果 `program_execution_rate < 0.95`，说明 Program schema 或 executor 兼容性有问题。
- 如果 `executed_answer_accuracy >= v2_sft2` 且输出更短，mixed v3 路线成立。
- 如果 `pass@8 - pass@1 >= 0.10`，说明后续 GRPO 仍有较好空间。
- 如果 `pass@1` 低于 existing v3，但 `pass@8` 更高，仍可能是更好的 RL 起点。
- 如果 `pass@1` 和 `pass@8` 都低于 existing v3，则暂时保留 existing v3 two-stage 为主 baseline。


## RL Handoff

本 notebook 不直接运行 GRPO，只产出 SFT policy checkpoint。

默认 mixed 路线的 RL 起点：

```text
/root/autodl-tmp/outputs/financial_reasoning_cot_pot/program_mixed_merged
```

后续可以在 `run_fingpt_cot_rl.ipynb` 中把 policy 初始化路径改成：

```python
SFT2_MERGED_OUT = Path('/root/autodl-tmp/outputs/financial_reasoning_cot_pot/program_mixed_merged')
```

进入 RL 前先看 benchmark：

- `pass@8 - pass@1 >= 0.10`：优先做 GRPO，把正确 Program 推向 greedy/high-probability。
- `pass@1` 已接近 `pass@8`：先不要急着 RL，可能 SFT 已经接近上限。
- execution reward 必须基于 strict Program parser/executor，避免自然语言 program 被错误执行。
